### W02B --- Three-state system

In [2]:
import numpy as np

In [3]:
class Node:
    def __init__(self , index , energy):
        self.index = index
        self.energy = energy

class System:
    def __init__(self, energies, kT, Tij):
        self.energies = energies
        self.kT = kT
        self.Tij = Tij
        self.nodes = [Node(i,e) for i,e in enumerate(self.energies)]
        self.state = self.nodes[0]
        self.partition_function = self.get_partition_function()
        self.probabilities = self.get_probabilities()
        self.thermal_avg = self.get_thermal_avg()

    def get_partition_function(self):
        return sum(np.exp(-self.energies/self.kT))
    def get_probabilities(self):
        return 1/self.partition_function * np.exp(-self.energies/self.kT)
    def get_thermal_avg(self):
        return sum(self.energies*self.probabilities)

    def proposed_state (self):
        p_states = self.Tij[self.state.index]
        j = np.random.choice([0,1,2], p=p_states)
        return self.nodes[j]
    
    def step(self):
        proposed_state = self.proposed_state()
        delta_e = proposed_state.energy - self.state.energy
        acceptance_probability = min(1, np.exp(-delta_e / self.kT))  # using P(x')/P(x) = exp(-E'/kT)/exp(E/kT) = exp(-delta_E/kT)
        if np.random.rand() < acceptance_probability:
            self.state = proposed_state

    def run_Metropolis_Monte_Carlo (self , n_steps):
        states = []
        for _ in range(n_steps):
            self.step ()
            states.append(self.state.index)
        counts = np.bincount(states)  # count the states
        mmc_prob = counts/len(states)  # get mmc probabilites
        return mmc_prob



trans_probs = np.array([[1/3, 1/3, 1/3], [1/3, 1/3, 1/3], [1/3, 1/3, 1/3]], dtype=float)
three_state = System(energies=np.array([0, -np.log(2), 0]), kT=1/3, Tij=trans_probs)
print(three_state.partition_function)
print(three_state.probabilities)
print(three_state.thermal_avg)

# using metropolis monte carlo and collecting all visited states:
mmc = three_state.run_Metropolis_Monte_Carlo(100000)
print(mmc)

10.000000000000002
[0.1 0.8 0.1]
-0.5545177444479562
[0.09894 0.80085 0.10021]


In [ ]:
varied_trans_probs = np.array([[1/2,1/2,0],[1/2,0,1/2],[0,1/2,1/2]])
print(varied_trans_probs)
varied_three_state = System(energies=np.array([0, -np.log(2), 0]), kT=1/3, Tij=varied_trans_probs)
mmc = varied_three_state.run_Metropolis_Monte_Carlo(100000)
print(mmc) 

# The symmetric matrix ensures that we still get the boltzmann distribution

[[0.5 0.5 0. ]
 [0.5 0.  0.5]
 [0.  0.5 0.5]]
[0.09744 0.80104 0.10152]


In [7]:
# Now what happens when using asymmetric probabilites?

asy_trans_probs = np.array([[8/10,1/10,1/10],[1/2,0,1/2],[0,1/2,1/2]])
print(asy_trans_probs)
asy_three_state = System(energies=np.array([0, -np.log(2), 0]), kT=1/3, Tij=asy_trans_probs)
mmc = asy_three_state.run_Metropolis_Monte_Carlo(100000)
print(mmc) 

# We no longer get the boltzmann distribution
# going from 0->1 has p=0.1, while 1->0 has p=0.5
# That skews the distribution

[[0.8 0.1 0.1]
 [0.5 0.  0.5]
 [0.  0.5 0.5]]
[0.2133  0.66285 0.12385]
